# Bull Flag Continuation Daily Notebook

这个 notebook 用来做四件事：

1. 用最新已收盘数据生成“明天可入场”的 bull flag 候选清单。
2. 用当前主版本 `BullFlagTrailingAfterTp1Researcher` 做每日信号筛选。
3. 读取你已经开仓的股票，并按动态 trailing 规则监控是否该在下一交易日开盘离场。
4. 快速复盘一笔当日最值得看的候选或离场提醒。

运行前请确认环境里已经设置好 `TUSHARE_TOKEN`。

In [1]:
# Notebook bootstrap: repo root + sys.path
import sys
from pathlib import Path

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "strategies").exists()
        and (path / "backtester").exists()
        and (path / "Dataframes").exists()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate repo root from the current notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

WindowsPath('C:/Users/Jay/GitRepo/codex_stock_pitch')

In [2]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

from strategies.china_stock_data import (
    get_csi500_member_prices,
    get_hs300_member_prices,
    get_next_trading_day,
)
from strategies.bull_flag_exit_variants import (
    BullFlagDynamicExitConfig,
    BullFlagTrailingAfterTp1Researcher,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [3]:
REPO_ROOT

WindowsPath('C:/Users/Jay/GitRepo/codex_stock_pitch')

In [4]:
STRATEGY_DIR = REPO_ROOT / "strategy_archive" / "bull_flag_continuation"
OUTPUT_DIR = STRATEGY_DIR / "outputs"
POSITIONS_PATH = STRATEGY_DIR / "open_positions.csv"

UNIVERSE = "csi500"  # "csi500" or "hs300"
LOOKBACK_CALENDAR_DAYS = 420
END_DATE = pd.Timestamp.today().normalize()
PAUSE_SECONDS = 0.0
MAX_CALLS_PER_MINUTE = 195
ENTRY_PRICE_BASIS = "follow_through_close"  # "follow_through_close" or "signal_close"
SAVE_OUTPUTS = True

cfg = BullFlagDynamicExitConfig(
    universe=UNIVERSE,
    max_flag_retrace_ratio=0.30,
    min_breakout_body_pct=0.60,
    max_breakout_upper_shadow_pct=0.35,
    max_breakout_lower_shadow_pct=0.50,
    max_peak_sma60_return_10=0.055,
    tp1_fraction_of_target=0.50,
    trailing_stop_fraction_of_flagpole=0.25,
)

if cfg.universe != UNIVERSE:
    raise ValueError("cfg.universe must match UNIVERSE.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
FETCHERS = {
    "csi500": get_csi500_member_prices,
    "hs300": get_hs300_member_prices,
}

def format_display_frame(
    df: pd.DataFrame,
    *,
    pct_columns: list[str] | None = None,
    price_columns: list[str] | None = None,
) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    formatted = df.copy()
    for column in pct_columns or []:
        if column in formatted.columns:
            formatted[column] = (pd.to_numeric(formatted[column], errors="coerce") * 100).round(2)
    for column in price_columns or []:
        if column in formatted.columns:
            formatted[column] = pd.to_numeric(formatted[column], errors="coerce").round(2)
    return formatted

def save_report(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"saved: {path}")


## 1. 拉取最新数据

这里会拉取当前指数最新成分股在最近一段时间内的前复权日线，并尽量拿到下一交易日日期。

In [7]:
TS_TOKEN = "4eea59000d8b3ca6ca39500a0c63cad66639bf093cdebed204a04271"

os.environ["TUSHARE_TOKEN"] = TS_TOKEN

if not os.environ.get("TUSHARE_TOKEN"):
    raise EnvironmentError("请先在环境变量里设置 TUSHARE_TOKEN。")

fetcher = FETCHERS[UNIVERSE]
start_date = END_DATE - pd.Timedelta(days=LOOKBACK_CALENDAR_DAYS)
price_df = fetcher(
    sd=start_date.date(),
    ed=END_DATE.date(),
    pause_seconds=PAUSE_SECONDS,
    max_calls_per_minute=MAX_CALLS_PER_MINUTE,
)

if price_df.empty:
    raise ValueError("price_df is empty. Check Tushare token, permissions, and date range.")

as_of_date = pd.Timestamp(price_df["date"].max()).normalize()
try:
    next_trade_date = get_next_trading_day(as_of_date)
except Exception as exc:
    warnings.warn(f"failed to resolve next trading day from Tushare calendar: {exc}")
    next_trade_date = as_of_date + pd.offsets.BDay(1)

failed_tickers = price_df.attrs.get("failed_tickers", [])
print(
    f"universe={UNIVERSE} | rows={len(price_df):,} | latest_bar={as_of_date.date()} | "
    f"next_trade_date={next_trade_date.date()}"
)
print(f"failed_tickers={len(failed_tickers)}")
if failed_tickers:
    print(failed_tickers[:20])


c:\Users\Jay\GitRepo\codex_stock_pitch\.venv\codex_stock_pitch\lib\site-packages\tushare\pro\data_pro.py:130: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['adj_factor'] = data['adj_factor'].fillna(method='bfill')
c:\Users\Jay\GitRepo\codex_stock_pitch\.venv\codex_stock_pitch\lib\site-packages\tushare\pro\data_pro.py:130: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['adj_factor'] = data['adj_factor'].fillna(method='bfill')
c:\Users\Jay\GitRepo\codex_stock_pitch\.venv\codex_stock_pitch\lib\site-packages\tushare\pro\data_pro.py:130: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['adj_factor'] = data['adj_factor'].fillna(method='bfill')
c:\Users\Jay\GitRepo\codex_stock_pitch\.venv\codex_stock_pitch\lib\site-pack

universe=csi500 | rows=139,666 | latest_bar=2026-04-17 | next_trade_date=2026-04-20
failed_tickers=0


## 2. 生成明日可入场候选

这里输出的是“明天开盘值得关注/准备入场”的 bull flag 名单，使用的是当前主版本的 entry 规则。

In [8]:
researcher = BullFlagTrailingAfterTp1Researcher(price_df, config=cfg)
next_candidates = researcher.get_next_session_candidates(
    as_of_date=as_of_date,
    next_trade_date=next_trade_date,
    entry_price_basis=ENTRY_PRICE_BASIS,
)

candidate_view = format_display_frame(
    next_candidates,
    pct_columns=[
        "flagpole_return",
        "flag_retrace_ratio",
        "signal_stack_spread_pct",
        "signal_sma20_return_5",
        "peak_sma60_return_10",
        "reward_to_risk",
    ],
    price_columns=[
        "entry_reference_price",
        "planned_hard_stop_price",
        "planned_take_profit_price",
        "flag_peak_high",
        "flag_low",
    ],
)

display(candidate_view)

if SAVE_OUTPUTS:
    save_report(next_candidates, OUTPUT_DIR / f"next_session_candidates_{as_of_date:%Y%m%d}.csv")


,signal_date,follow_through_date,planned_entry_date,ticker,ts_code,name,entry_price_basis,entry_reference_price,planned_hard_stop_price,planned_take_profit_price,flag_peak_high,flag_low,flagpole_return,flag_retrace_ratio,flag_bars,signal_bullish_stack_run_length,signal_stack_spread_pct,signal_sma20_return_5,peak_bullish_stack_run_length,peak_sma60_return_10,reward_to_risk,follow_through_confirmed,follow_through_close_gt_signal_close,trend_environment_ok,reward_to_risk_ok,entry_signal_live


saved: C:\Users\Jay\GitRepo\codex_stock_pitch\strategy_archive\bull_flag_continuation\outputs\next_session_candidates_20260417.csv


## 3. 读取持仓并监控动态退出

`open_positions.csv` 至少需要三列：`ticker`、`entry_date`、`entry_price`。
可选列有：`shares`、`signal_date`、`note`。

In [11]:
if not POSITIONS_PATH.exists():
    pd.DataFrame(columns=["ticker", "entry_date", "entry_price", "shares", "signal_date", "note"]).to_csv(
        POSITIONS_PATH,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"created empty positions file: {POSITIONS_PATH}")

open_positions = pd.read_csv(POSITIONS_PATH)
for column in ["entry_date", "signal_date"]:
    if column in open_positions.columns:
        open_positions[column] = pd.to_datetime(open_positions[column], errors="coerce")

display(open_positions)


,ticker,entry_date,entry_price,shares,signal_date,note


In [12]:
if open_positions.empty:
    position_monitor = pd.DataFrame()
    exit_alerts = pd.DataFrame()
    hold_watchlist = pd.DataFrame()
    print("open_positions.csv is empty. Fill it after you open positions.")
else:
    position_monitor = researcher.monitor_positions(
        open_positions,
        as_of_date=as_of_date,
        next_trade_date=next_trade_date,
    )
    exit_alerts = position_monitor[position_monitor["action"].isin(["exit_next_open", "exit_overdue"])].copy()
    hold_watchlist = position_monitor[position_monitor["action"].eq("hold")].copy()

    display(
        format_display_frame(
            exit_alerts,
            pct_columns=["pnl_pct", "reward_to_risk"],
            price_columns=[
                "entry_price",
                "current_close",
                "hard_stop_price",
                "take_profit_price",
                "tp1_price",
                "active_protective_stop",
                "post_tp1_stop_price",
                "ma_trail_value",
                "structure_trail_value",
                "close_retrace_threshold",
                "flag_peak_high",
                "flag_low",
            ],
        )
    )
    display(
        format_display_frame(
            hold_watchlist,
            pct_columns=["pnl_pct", "reward_to_risk"],
            price_columns=[
                "entry_price",
                "current_close",
                "hard_stop_price",
                "take_profit_price",
                "tp1_price",
                "active_protective_stop",
                "post_tp1_stop_price",
                "ma_trail_value",
                "structure_trail_value",
                "close_retrace_threshold",
                "flag_peak_high",
                "flag_low",
            ],
        )
    )

    if SAVE_OUTPUTS:
        save_report(position_monitor, OUTPUT_DIR / f"position_monitor_{as_of_date:%Y%m%d}.csv")
        save_report(exit_alerts, OUTPUT_DIR / f"exit_alerts_{as_of_date:%Y%m%d}.csv")
        save_report(hold_watchlist, OUTPUT_DIR / f"hold_watchlist_{as_of_date:%Y%m%d}.csv")


open_positions.csv is empty. Fill it after you open positions.


## 4. 快速复盘一笔最值得看的样本

默认优先复盘当日候选里的第一笔；如果今天没有新候选，就复盘第一笔离场提醒。

In [13]:
review_source = ""
review_ticker = None
review_signal_date = None

if not next_candidates.empty:
    review_source = "candidate"
    review_ticker = str(next_candidates.iloc[0]["ticker"])
    review_signal_date = pd.Timestamp(next_candidates.iloc[0]["signal_date"])
elif not exit_alerts.empty:
    review_source = "exit_alert"
    review_ticker = str(exit_alerts.iloc[0]["ticker"])
    review_signal_date = pd.Timestamp(exit_alerts.iloc[0]["signal_date_resolved"])

if review_ticker is None:
    print("今天没有可复盘的候选或离场提醒。")
else:
    inspection = researcher.inspect_signal(review_ticker, review_signal_date, lookback=80, lookahead=20)
    print(f"review_source={review_source} | ticker={review_ticker} | signal_date={review_signal_date.date()}")
    display(pd.DataFrame([inspection["summary"]]))
    display(inspection["condition_checklist"])
    exit_path = inspection.get("exit_path", pd.DataFrame())
    if not exit_path.empty:
        display(exit_path.tail(15))
    inspection["plot"] = researcher.plot_signal_context(review_ticker, review_signal_date, lookback=80, lookahead=20)
    inspection["plot"]


今天没有可复盘的候选或离场提醒。


In [14]:
summary = pd.DataFrame(
    [
        {
            "as_of_date": as_of_date,
            "next_trade_date": next_trade_date,
            "candidate_count": len(next_candidates),
            "exit_alert_count": len(exit_alerts),
            "hold_count": len(hold_watchlist),
            "failed_ticker_count": len(failed_tickers),
            "strategy_name": researcher.STRATEGY_NAME,
        }
    ]
)
display(summary)


,as_of_date,next_trade_date,candidate_count,exit_alert_count,hold_count,failed_ticker_count,strategy_name
0,2026-04-17,2026-04-20,0,0,0,0,bull_flag_trailing_after_tp1
